# Car Price Prediction — Feature Based

**Tabular Regression Project** — Predict house values using feature-based approaches.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: California Housing (sklearn, 20,640 rows × 8 features)
Target: `MedHouseVal`

## 2 · Project Overview

This project uses the **California Housing** dataset from sklearn to demonstrate **feature-based prediction**. We focus on systematic feature engineering, selection, and model comparison. The dataset has 20,640 rows — large enough for robust model training.

*Note: Despite the folder name referencing 'Car Price', the underlying pipeline uses the California Housing dataset as a proxy for feature-based regression.*

## 3 · Learning Objectives

1. Load datasets from sklearn's built-in collection.
2. Apply systematic feature engineering.
3. Compare feature importance across model families.
4. Use LazyPredict and FLAML for benchmarking.
5. Evaluate with RMSE, MAE, and R².

## 4 · Problem Statement

Predict **median house value** in California districts using 8 demographic and geographic features. Focus on which features matter most and how engineering new features improves predictions.

## 5 · Why This Project Matters

- Demonstrates that **feature engineering** can matter more than model choice.
- California Housing is a well-understood benchmark for regression.
- Teaches sklearn's dataset loading workflow.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 20,640 |
| **Features** | MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude |
| **Target** | MedHouseVal (median house value, $100k units) |
| **Source** | `sklearn.datasets.fetch_california_housing` |

## 7 · Dataset Source and License Notes

- **Source**: 1990 California Census, via sklearn.
- **License**: Public domain.
- **Note**: Values are in units of $100,000.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'MedHouseVal'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')
print(f'Target mean: {df[TARGET].mean():.2f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df['MedInc'], df[TARGET], alpha=0.1, s=3)
axes[0,0].set_xlabel('Median Income'); axes[0,0].set_ylabel(TARGET)
axes[0,0].set_title('Income vs House Value')

axes[0,1].scatter(df['Longitude'], df['Latitude'], c=df[TARGET], cmap='viridis', alpha=0.2, s=3)
axes[0,1].set_xlabel('Longitude'); axes[0,1].set_ylabel('Latitude')
axes[0,1].set_title('Geographic Price Distribution')

df[TARGET].hist(bins=50, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('House Value Distribution')

corr = df.corr()[TARGET].drop(TARGET).sort_values()
corr.plot(kind='barh', ax=axes[1,1])
axes[1,1].set_title('Correlation with Target')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Capped at 5.0: {(df[TARGET] >= 5.0).sum()} rows')

## 15 · Train / Validation / Test Split

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are numeric. No missing values or encoding needed.

## 17 · Feature Engineering

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rooms_per_household'] = df_part['AveRooms'] / (df_part['AveOccup'] + 0.01)
    df_part['bedrooms_ratio'] = df_part['AveBedrms'] / (df_part['AveRooms'] + 0.01)
    df_part['pop_per_household'] = df_part['Population'] / (df_part['AveOccup'] + 0.01)
    df_part['income_per_age'] = df_part['MedInc'] / (df_part['HouseAge'] + 1)

print(f'Features after engineering: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.4f}, MAE={baseline_mae:.4f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
X_tr_lp = X_train.head(5000); y_tr_lp = y_train.head(5000)
X_te_lp = X_test.head(1000); y_te_lp = y_test.head(1000)
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_tr_lp, X_te_lp, y_tr_lp, y_te_lp)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **MedInc** (median income) dominates — income is the single best predictor of house value.
- **Location** (lat/lon) captures neighborhood premium.
- Engineered per-household features add valuable density information.
- Feature importance analysis shows which features a model truly relies on.

## 26 · Limitations

- 1990 Census data.
- Target capped at $500K.
- District-level, not property-level.
- No property-specific features.

## 27 · How to Improve This Project

1. Cluster lat/lon into neighborhoods.
2. Log-transform the target.
3. Add ocean proximity feature.
4. Hyperparameter tune with Optuna.

## 28 · Production Considerations

- Need current data.
- Property-level features for real valuations.
- Confidence intervals for lending.
- Regular retraining.

## 29 · Common Mistakes

1. Using AveRooms directly without per-household normalization.
2. Ignoring capped target values.
3. Not leveraging geographic features.
4. Over-engineering on numeric-only data.

## 30 · Mini Challenge / Exercises

1. Plot prediction errors on a lat/lon map.
2. Try target encoding lat/lon bins.
3. Remove the bottom 5 features by importance and retrain.
4. Compare log(target) vs raw target R².

## 31 · Final Summary / Key Takeaways

- Feature engineering (per-household ratios) improves all models.
- Income and location are the dominant predictors.
- Boosting models significantly outperform linear regression.
- Systematic feature importance analysis guides future feature collection.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')